In [2]:
from inro.emme.database.scenario import Scenario as EmmeScenario
from inro.emme.network import Network
from inro.emme.database.emmebank import Emmebank, create as _create_emmebank
from shapely.geometry import mapping, LineString
import json
from typing import Dict, List, Optional
import inro.modeller as _m

In [3]:
transit_bank = Emmebank('../Database_transit/emmebank')
highway_bank = Emmebank('../Database_highway/emmebank')

In [8]:
scenario = bank.scenario(12)

In [11]:
scenario.element_totals

{'centroids': 4756,
 'regular_nodes': 477431,
 'links': 1602820,
 'turn_entries': 0,
 'transit_lines': 1252,
 'transit_segments': 231425,
 'turns': 0,
 'modes': 17,
 'transit_vehicles': 119}

In [12]:
transit_network = scenario.get_network()

In [24]:
output_path = "E:\\TM2\\2015_TM2_20250619\\output_summaries\\boardings_by_segment_am.csv"
with open(output_path, 'w') as f:
    f.write(
    ",".join([
        "Line", 
        "From",
        "To",
        "Length", 
        "Dwt",
        "capt",
        "TTF",
        "voltr",
        "caps",
        "Data1",
        "Data2",
        "Data3"
    ])
    )
    f.write("\n")
    
    for line in transit_network.transit_lines():
        total_capacity = line.vehicle.total_capacity
        seated_capacity = line.vehicle.seated_capacity
        hdw = line.headway
        line_hour_total_cap = 60 * total_capacity / hdw
        line_hour_seated_cap = 60 * seated_capacity / hdw
        for segment in line.segments(include_hidden=False):
            f.write(
                ",".join(
                    [
                        str(x) 
                        for x in [
                            segment.line.id, 
                            segment.i_node, 
                            segment.j_node,
                            segment.link.length,  
                            segment.dwell_time,
                            line_hour_total_cap,
                            segment.transit_time_func,
                            segment.transit_volume,
                            line_hour_seated_cap,
                            segment.data1,
                            segment.data2,
                            segment.data3
                        ]
                    ]
                )
            )
            f.write("\n")

In [15]:
output_path_geojson = "E:\\TM2\\2015_TM2_20250619\\output_summaries\\boardings_by_segment_am.geojson"

In [23]:
features = []
for line in transit_network.transit_lines():
    total_capacity = line.vehicle.total_capacity
    seated_capacity = line.vehicle.seated_capacity
    hdw = line.headway
    line_hour_total_cap = 60 * total_capacity / hdw
    line_hour_seated_cap = 60 * seated_capacity / hdw

    for segment in line.segments(include_hidden=False):
        geometry = mapping(LineString(segment.link.shape))
        feature = {
            "type": "Feature",
            "geometry": geometry,
            "properties": {
                "LINE_ID": segment.line.id,
                "INODE": int(segment.i_node.id),
                "JNODE": int(segment.j_node.id),
                "VOLTR": segment.transit_volume,
                "caps": line_hour_seated_cap,
                "capt": line_hour_total_cap
            }
        }
        features.append(feature)

geojson_data = {
    "type": "FeatureCollection",
    "crs": {
        "type": "name",
        "properties": {"name": "urn:ogc:def:crs:EPSG::2875"},
    },
    "features": features
}

with open(output_path_geojson, "w") as f:
    json.dump(geojson_data, f, indent=2)

In [76]:
transit_attributes = {
    "LINK": ['#link_id', "@trantime", "@ft"],
    "TRANSIT_SEGMENT": ["@schedule_time", "@trantime_seg", "data1"]
}

In [70]:
def copy_attribute_values(src, 
                          dst, 
                           src_attributes: Dict[str, List[str]],
                        dst_attributes: Optional[Dict[str, List[str]]] = None,
                         ):
    for domain, src_attrs in src_attributes.items():
        if src_attrs:
            dst_attrs = src_attrs
            if dst_attributes is not None:
                dst_attrs = dst_attributes.get(domain, src_attrs)
            values = src.get_attribute_values(domain, src_attrs)
            dst.set_attribute_values(domain, dst_attrs, values)

In [77]:
copy_attribute_values(scenario, transit_network, transit_attributes)

In [80]:
transit_link_dict = {
            tran_link["#link_id"]: tran_link for tran_link in transit_network.links()
        }

In [81]:
transit_link_dict['#link_id']['i_node']

KeyError: '#link_id'

In [82]:
transit_link_dict

{0: Link(482187-477842),
 77108: Link(1-6345),
 78851: Link(1-24987),
 79119: Link(1-27779),
 77532: Link(2-11163),
 78081: Link(2-16351),
 78144: Link(2-17005),
 77248: Link(3-7750),
 77580: Link(3-11662),
 77583: Link(3-11677),
 79143: Link(3-28063),
 77970: Link(4-15283),
 78532: Link(4-21458),
 78605: Link(4-22191),
 77581: Link(5-11662),
 78331: Link(5-19222),
 78824: Link(5-24659),
 78884: Link(5-25265),
 78444: Link(6-20456),
 78783: Link(6-24353),
 77025: Link(7-5465),
 77212: Link(7-7419),
 78885: Link(7-25265),
 77930: Link(8-14924),
 78179: Link(8-17405),
 78837: Link(8-24845),
 77724: Link(9-13035),
 78273: Link(9-18411),
 78866: Link(9-25106),
 77126: Link(10-6532),
 77911: Link(10-14761),
 77955: Link(10-15118),
 78011: Link(10-15673),
 77927: Link(11-14912),
 77937: Link(11-14934),
 77431: Link(12-9904),
 77743: Link(12-13190),
 79115: Link(12-27744),
 77005: Link(13-5163),
 77806: Link(13-13760),
 78329: Link(13-19204),
 78671: Link(13-23028),
 77958: Link(14-15162),
 7

In [39]:
attributes = transit_network.get_attribute_values()

In [42]:
attributes.names

AttributeError: 'function' object has no attribute 'names'

### Export Network as shapefiles

In [4]:
### Export network as shapefiles
_MODELLER = _m.Modeller()
network_to_shapefile = _MODELLER.tool("inro.emme.data.network.export_network_as_shapefile")

In [96]:
## Output transit shapefile
for i in range(11,16):
    scenario = transit_bank.scenario(i)
    network_to_shapefile(
        export_path = "../../output_summaries/Scenario_{i}",
        scenario = scenario,
        transit_shapes = 'LINES_AND_SEGMENTS',
        selection = {
            "link":'none',
            "node":'none',
            "turn": 'none',
            'transit_line': 'all'
        }
    )

{'field_mapping': {'nodes': OrderedDict([('ID', 'ID'),
               ('X', 'X'),
               ('Y', 'Y'),
               ('DATA1', 'DATA1'),
               ('DATA2', 'DATA2'),
               ('DATA3', 'DATA3'),
               ('ISZONE', 'ISZONE'),
               ('ISINTERSEC', 'ISINTERSEC'),
               ('LABEL', 'LABEL'),
               ('INBOAI', 'INBOAI'),
               ('FIALII', 'FIALII'),
               ('@bike_node', '@bike_node'),
               ('@drive_node', '@drive_nod'),
               ('@farezone', '@farezone'),
               ('@hdw_fraction', '@hdw_fract'),
               ('@maz_id', '@maz_id'),
               ('@rail_node', '@rail_node'),
               ('@stop_tap_id', '@stop_tap_'),
               ('@tap_id', '@tap_id'),
               ('@taz_id', '@taz_id'),
               ('@wait_pfactor', '@wait_pfac'),
               ('@walk_node', '@walk_node'),
               ('@xboard_nodepen', '@xboard_no'),
               ('#node_county', '#node_coun'),
              

In [7]:
# Output hwy shapefile
for i in range(11,16):
    print(f'Processing scenario {i}')
    scenario = highway_bank.scenario(i)
    network_to_shapefile(
        export_path = f"../../output_summaries/Scenario_{i}",
        scenario = scenario,
        selection = {
            "link":'all',
            "node":'all',
            "turn": 'all',
            'transit_line': 'none'
        }
    )

Processing scenario 11
Processing scenario 12
Processing scenario 13
Processing scenario 14
Processing scenario 15


In [98]:
scenario.has_traffic_results

False

In [99]:
highway_bank = Emmebank('../Database_highway/emmebank')
hwy_scenario = highway_bank.scenario(12)

In [100]:
hwy_scenario.has_traffic_results

True